# CocktailCompass: A Chunked Cocktail RAG Assistant

This notebook follows a transparent RAG pipeline: cleaned cocktail records → overlapping text chunks → embeddings → FAISS → retrieved evidence → answer.

## 1. Install the small set of dependencies

In [ ]:
%pip install -q datasets sentence-transformers faiss-cpu transformers sentencepiece
print('Dependencies installed.')

## 2. Import libraries and set configuration

In [ ]:
import ast
import re
import textwrap
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

PROJECT_DIR = Path('/content/cocktailcompass')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = 'erwanlc/cocktails_recipe'
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
GENERATOR_MODEL_NAME = 'google/flan-t5-base'

# A simple, explicit chunking policy for the assessment report.
CHUNK_SIZE_WORDS = 120
CHUNK_OVERLAP_WORDS = 20
TOP_K = 3
DISTANCE_THRESHOLD = 1.15

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Chunk size: {CHUNK_SIZE_WORDS} words | overlap: {CHUNK_OVERLAP_WORDS} words')

## 3. Load the cocktail dataset

In [ ]:
dataset = load_dataset(DATASET_NAME, split='train')
raw_df = dataset.to_pandas()
print(f'Number of source records: {len(raw_df):,}')
display(raw_df.head(3))

## 4. Clean records and create overlapping recipe chunks

In [ ]:
def clean_text(value, default=''):
    if pd.isna(value):
        return default
    text = str(value).strip()
    # Repair only common UTF-8-as-latin-1 mojibake when it is present.
    if any(marker in text for marker in ('Ã', 'Â', 'â')):
        try:
            text = text.encode('latin-1').decode('utf-8')
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    return text


def parse_ingredients(value):
    try:
        pairs = ast.literal_eval(str(value))
        return '; '.join(f'{quantity} {ingredient}' for quantity, ingredient in pairs)
    except (ValueError, SyntaxError):
        return clean_text(value)


def split_into_chunks(text, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    words = text.split()
    if not words:
        return []
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(' '.join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return chunks


# Remove only rows with no title or recipe, and exact duplicate rows.
cocktails_df = raw_df.copy()
cocktails_df = cocktails_df.dropna(subset=['title', 'recipe']).drop_duplicates().reset_index(drop=True)

chunks = []
for record_id, row in cocktails_df.iterrows():
    title = clean_text(row['title'])
    ingredients = parse_ingredients(row['ingredients'])
    recipe = clean_text(row['recipe'])
    glass = clean_text(row['glass'], default='Not specified')
    garnish = clean_text(row['garnish'], default='Not specified')

    # The source metadata is repeated in every chunk for grounded retrieval.
    metadata = (
        f'Cocktail: {title}\n'
        f'Ingredients: {ingredients}\n'
        f'Glass: {glass}\n'
        f'Garnish: {garnish}\n'
    )
    recipe_chunks = split_into_chunks(recipe)
    for chunk_number, recipe_chunk in enumerate(recipe_chunks, start=1):
        chunks.append({
            'record_id': int(record_id),
            'title': title,
            'chunk_number': chunk_number,
            'ingredients': ingredients,
            'glass': glass,
            'garnish': garnish,
            'recipe_chunk': recipe_chunk,
            'text': metadata + f'Recipe chunk {chunk_number}: {recipe_chunk}',
        })

chunk_lengths = pd.Series([len(chunk['recipe_chunk'].split()) for chunk in chunks])
print(f'Cleaned cocktail records: {len(cocktails_df):,}')
print(f'Recipe chunks created: {len(chunks):,}')
print('Chunk length summary (words):')
display(chunk_lengths.describe().to_frame(name='recipe_chunk_words'))
print(chunks[0]['text'][:700])

## 5. Embed the recipe chunks and build the FAISS index

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
chunk_texts = [chunk['text'] for chunk in chunks]
chunk_embeddings = embedding_model.encode(chunk_texts, show_progress_bar=True)
chunk_embeddings = np.asarray(chunk_embeddings, dtype='float32')

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print(f'Embedding dimension: {dimension}')
print(f'Chunks stored in FAISS: {index.ntotal:,}')

## 6. Load the open-weight generator model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
generator_model = AutoModelForSeq2SeqLM.from_pretrained(GENERATOR_MODEL_NAME).to(device)
generator_model.eval()
print(f'Generator model: {GENERATOR_MODEL_NAME}')

## 7. Define a grounded RAG function with title-first retrieval

In [ ]:
def find_exact_title_chunks(question):
    question_lower = question.lower()
    exact_chunks = []
    for chunk in chunks:
        title_pattern = rf'(?<!\w){re.escape(chunk["title"].lower())}(?!\w)'
        if re.search(title_pattern, question_lower):
            exact_chunks.append(chunk)
    return exact_chunks


def ask_rag(question, top_k=TOP_K, distance_threshold=DISTANCE_THRESHOLD):
    # Use an exact cocktail-title match first. FAISS is the fallback.
    exact_title_chunks = find_exact_title_chunks(question)
    if exact_title_chunks:
        retrieved_chunks = [(chunk, 0.0, 'exact title') for chunk in exact_title_chunks[:top_k]]
    else:
        question_embedding = embedding_model.encode([question])
        question_embedding = np.asarray(question_embedding, dtype='float32')
        distances, indices = index.search(question_embedding, top_k)
        retrieved_chunks = []
        for chunk_index, distance in zip(indices[0], distances[0]):
            if distance <= distance_threshold:
                retrieved_chunks.append((chunks[chunk_index], float(distance), 'FAISS'))

    print('QUESTION:')
    print(question)
    print('\nRETRIEVED CHUNKS:')

    if not retrieved_chunks:
        print('No sufficiently relevant chunks found.')
        print('\nANSWER:')
        print('The retrieved documents do not provide enough information.')
        return

    for chunk, distance, route in retrieved_chunks:
        print(
            f"- {chunk['title']} | chunk {chunk['chunk_number']} | "
            f"route: {route} | distance: {distance:.4f}"
        )

    context = '\n\n'.join(
        f"Source title: {chunk['title']} | Recipe chunk {chunk['chunk_number']}\n{chunk['text']}"
        for chunk, _, _ in retrieved_chunks
    )

    prompt = f"""
You are a cocktail recipe assistant.

Answer the user's question using only the retrieved cocktail chunks below.

Rules:
1. Use only the retrieved chunks as the source of truth.
2. If the answer is not explicitly supported, say: The retrieved documents do not provide enough information.
3. Do not invent quantities, ingredients, methods, glassware, garnish, or origins.
4. Name the source cocktail title in the answer.
5. Keep the answer concise and in English.

Retrieved chunks:
{context}

User question:
{question}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)
    with torch.inference_mode():
        outputs = generator_model.generate(
            **inputs,
            max_new_tokens=110,
            num_beams=1,
            do_sample=False,
            no_repeat_ngram_size=3,
            repetition_penalty=1.15,
        )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print('\nANSWER:')
    print(textwrap.fill(answer, width=100))

## 8. Try example questions and inspect the retrieved chunks

In [ ]:
ask_rag('How do I make a Mojito?')

# Try other questions:
# ask_rag('Which cocktails use rum and lime?')
# ask_rag('How do I make a refreshing cocktail with ginger?')
# ask_rag('What is the penalty for late submission?')